# Synthetic Maser Data: Visualization and Initial Analysis

This notebook (1) plots the synthetic datasets and (2) walks through the baseline
method-comparison analysis. It is a **worked example**: the findings are
*illustrative* and rest on arbitrary modelling choices. Your job is to stress-test
and extend them (see `docs/SYNTHETIC_DATA_GUIDE.md`, "Open questions to chase").

**Firewall:** nothing in this notebook touches the real maser labels. The
generator draws from the real feature values only (the X-ray and WISE columns)
and invents the labels itself from the chosen scenario.

In [ ]:
import sys, os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
for p in ("src", "../src"):
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.insert(0, ap)

import numpy as np
import matplotlib.pyplot as plt
import synth_data as sd
import synth_analysis as sa
from maser_data import FEATURES, TARGET
%matplotlib inline

## 1. The synthetic data

`make_dataset` returns a **pandas DataFrame whose columns are named exactly as in
the real data** (`maser_data.py`), so any pipeline you write here runs unchanged
on the real frames. It also carries two synthetic-only diagnostic columns,
`z_true` (the latent true label) and `detected`, which the real data cannot give
you.

In [ ]:
df = sd.make_dataset(plane="xray", scenario="wedge", seed=0)
print("columns:", list(df.columns))
print("synthetic dataset configuration:")
for key, value in df.attrs.items():
    if isinstance(value, (float, np.floating)):
        value = round(float(value), 4)
    print(f"  {key}: {value}")
df.head()

### Scenarios as pictures

The `scenario` argument sets the *true* decision boundary. Red = a maser
(observed target), grey = nonmaser. Positives sit in the **high-risk corner**,
which the `signal_signs` orientation fixes a priori from the physics: for the
X-ray plane that is high `L_12um_bestfit_1` (mid-IR luminous) and low `Lob`
(X-ray faint, i.e. obscured), so the masers cluster toward the lower-right here.
`wedge`/`box` make a hard corner there, `interaction` tilts it, and `blob` makes
a central island. Feature positions come from a jittered bootstrap of the real
rows (the default `feature_model`: real feature rows resampled with a little
random jitter), so the cloud keeps the real skew and gaps.

In [ ]:
scenarios = ["linear", "wedge", "box", "interaction", "blob"]
xcol, ycol = FEATURES["xray"]
tgt = TARGET["xray"]

fig, axes = plt.subplots(1, len(scenarios), figsize=(20, 4),
                         sharex=True, sharey=True)
for ax, sc in zip(axes, scenarios):
    d = sd.make_dataset("xray", scenario=sc, strength=3.0, seed=0)
    neg, pos = d[d[tgt] == 0], d[d[tgt] == 1]
    ax.scatter(neg[xcol], neg[ycol], s=6, c="lightgray", label="nonmaser")
    ax.scatter(pos[xcol], pos[ycol], s=16, c="firebrick", label="maser")
    ax.set_title(sc)
    ax.set_xlabel(xcol)
axes[0].set_ylabel(ycol)
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

### The WISE plane and the fused features

The WISE-only plane has its own two colours (and many more positives, ~114). The
fused dataset stacks X-ray and WISE columns; the right panel shows the WISE
colour `w1w2` is strongly correlated with the 12 micron mid-IR luminosity
`L_12um_bestfit_1` in the real data (r ~ 0.82), which is why "does WISE add
anything beyond the X-ray features?" is a genuine question.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

dw = sd.make_dataset("wise", scenario="wedge", strength=3.0, seed=0)
wx, wy = FEATURES["wise"]; wt = TARGET["wise"]
ax[0].scatter(dw[dw[wt] == 0][wx], dw[dw[wt] == 0][wy], s=4, c="lightgray")
ax[0].scatter(dw[dw[wt] == 1][wx], dw[dw[wt] == 1][wy], s=12, c="firebrick")
ax[0].set_xlabel(wx); ax[0].set_ylabel(wy)
ax[0].set_title("WISE plane (wedge truth)")

dfu = sd.make_fused_dataset(scenario="linear", wise_signal=0.0, seed=0)
ax[1].scatter(dfu["L_12um_bestfit_1"], dfu["wise_w1w2"], s=8, c="steelblue")
ax[1].set_xlabel("L_12um_bestfit_1"); ax[1].set_ylabel("wise_w1w2")
ax[1].set_title("WISE colour vs 12um mid-IR luminosity (correlated)")
plt.tight_layout()
plt.show()

## 2. Baseline analysis (illustrative)

Now we evaluate how the candidate models behave. Every number below depends on
the arbitrary truth we simulated, so treat each as a hypothesis to confirm or
break across several settings, not a fact.

### Is there anything to detect? (effect ceiling at large n)

Fit each model on a huge sample so noise disappears, and measure how much one
truly beats the other. If this is ~0, no finite-sample test can find a
difference, because there genuinely is none. These large-n checks use the GMM
feature sampler so the synthetic cloud is smooth rather than many jittered
copies of the empirical rows.

In [ ]:
XF, XT = FEATURES["xray"], TARGET["xray"]
print("Large-n effect ceiling (test set n=40,000; feature_model='gmm')")
print("Values are delta AUC: candidate model AUC minus plain-LR AUC.")
print(f"{'scenario':12s} {'interaction_lr_delta_auc':>25s} {'quadratic_lr_delta_auc':>25s}")
for sc in ["linear", "wedge", "box", "interaction", "blob"]:
    big = sa.maker("xray", scenario=sc, n=40000, strength=3.0, feature_model="gmm")
    ci = sa.effect_ceiling(big, (sa.interaction_lr, XF), (sa.plain_lr, XF), XT)
    cq = sa.effect_ceiling(big, (sa.quadratic_lr, XF), (sa.plain_lr, XF), XT)
    print(f"{sc:12s} {ci['dAUC']:>+25.3f} {cq['dAUC']:>+25.3f}")

**Reading it:** with the smooth GMM feature cloud, `wedge` (~+0.08) and even
the hard `box` (~+0.03) give the interaction model a small but real edge at
infinite n; `interaction` and `blob` give more. How large these effects are
depends on the shape of the feature cloud, so compare against
`feature_model="bootstrap"` and `"gaussian"` before treating the pattern as
robust.

### Power: can we detect it at the real sample size?

Repeat the full cross-validated comparison over many simulated datasets and ask
how often the decision rule fires. Here `power_detected` means the fraction of
simulated datasets where the corrected CI excludes zero **and** the mean gain is
at least 2 extra masers in a full-sample top-50 campaign. Run under `linear` for
the false-positive rate, under a real effect for power.

In [ ]:
power_specs = [
    ("linear", sa.interaction_lr, "null -> false-positive rate"),
    ("wedge", sa.interaction_lr, "soft corner"),
    ("box", sa.interaction_lr, "hard corner"),
    ("interaction", sa.interaction_lr, "multiplicative interaction"),
    ("blob", sa.quadratic_lr, "central island (positive control)"),
]
for sc, build_a, note in power_specs:
    pw = sa.power_study(sa.maker("xray", scenario=sc, strength=3.0),
                        (build_a, XF), (sa.plain_lr, XF), XT, n_sims=30)
    print(f"truth={sc:12s} power_detected={pw['rate']:.2f}  "
          f"mean_extra_masers_top{pw['campaign']}={pw['mean_effect_masers']:+.2f}"
          f"  ({note})")

**Reading it:** at the pre-registered operating point (+2 masers in a
full-sample top-50 campaign), the nonlinear comparisons have real power at n=641:
wedge ~0.7, box ~0.8, the explicit interaction ~0.4, and the blob positive
control ~1.0, while the linear null almost never fires. The effect is measured at
the full-sample campaign scale (top-50), not a per-fold top-10; those differ by
the number of folds, so be explicit about which scale a threshold refers to.

### Beating the hand-drawn cut (the headline comparison)

The paper selects candidates with fixed, hand-drawn cuts: Kuo's combined
inferred-N_H and L12 cut on the X-ray plane, and the Stern W1-W2 >= 0.8 colour
cut on WISE. These cuts are hard inside/outside rules, not probability scores.
For each held-out fold, we compute the cut's precision and recall, then evaluate
the fitted model at the same recall by choosing the model threshold with highest
precision subject to recall >= the cut's recall. Here the effect is converted to
extra masers at the cut's own selected-count operating point, not at top-50.

In [ ]:
print("X-ray: fitted interaction-LR vs the Kuo N_H>=24 and L12>1e42 cut")
for sc in ["wedge", "box", "interaction"]:
    pw = sa.cut_power_study(sa.maker("xray", scenario=sc, strength=3.0),
                            (sa.interaction_lr, XF), (sa.xray_cut, XF), XT,
                            n_sims=30)
    print(f"  truth={sc:12s} power_detected={pw['rate']:.2f}  "
          f"mean_extra_masers_at_cut={pw['mean_effect_masers']:+.2f}  "
          f"mean_cut_selected={pw['mean_selected']:.1f} galaxies")

WF, WT = FEATURES["wise"], TARGET["wise"]
print("WISE: fitted interaction-LR vs the Stern W1-W2>=0.8 cut")
for sc in ["wedge", "interaction"]:
    pw = sa.cut_power_study(sa.maker("wise", scenario=sc, strength=3.0),
                            (sa.interaction_lr, WF), (sa.stern_cut, WF), WT,
                            n_sims=30)
    print(f"  truth={sc:12s} power_detected={pw['rate']:.2f}  "
          f"mean_extra_masers_at_cut={pw['mean_effect_masers']:+.2f}  "
          f"mean_cut_selected={pw['mean_selected']:.1f} galaxies")

**Reading it:** this checks whether the model-vs-cut comparison has power when
the synthetic truth is not exactly the paper's cut. The reported effect is the
precision gain converted into extra masers at the cut's own selected-count
operating point (`mean_cut_selected`), not at a fixed top-k budget. The real A1/C1 comparison
runs on the actual labels after the plan is frozen; this synthetic section only
checks whether the comparison machinery behaves sensibly.

### Fusion: does adding WISE colours to X-ray help?

Same model (LR), two feature sets: the 4 fused columns vs the 2 X-ray columns.
`wise_signal=0` is a world where WISE is redundant; `wise_signal=1.5` is a world
where WISE carries real independent information.

In [ ]:
for ws, label in [(0.0, "redundant WISE"), (1.5, "WISE adds signal")]:
    arm_a = (sa.plain_lr, FEATURES["fused"])
    arm_b = (sa.plain_lr, FEATURES["xray"])
    big = sa.fused_maker(scenario="linear", wise_signal=ws, n=40000,
                         strength=3.0, feature_model="gmm")
    ceil = sa.effect_ceiling(big, arm_a, arm_b, TARGET["fused"])
    pw = sa.power_study(sa.fused_maker(scenario="linear", wise_signal=ws,
                        strength=3.0), arm_a, arm_b, TARGET["fused"], n_sims=30)
    print(f"wise_signal={ws} ({label:16s}): "
          f"large_n_delta_auc={ceil['dAUC']:+.3f}  "
          f"power_detected_at_n602={pw['rate']:.2f}  "
          f"mean_extra_masers_top{pw['campaign']}={pw['mean_effect_masers']:+.2f}")

**Reading it:** the fusion test behaves well: it correctly finds nothing
when WISE is redundant and detects the gain reliably when WISE adds real signal.
So "does WISE help?" is worth keeping as a confirmatory comparison.

### Optimism: which models flatter themselves at small n?

Resubstitution AUC (scored on the training data) minus honest cross-validated
AUC. The gap is how much a model overstates itself, and it grows with
flexibility.

In [ ]:
print("Optimism = training/resubstitution AUC minus cross-validated AUC.")
mk = sa.maker("xray", scenario="interaction", strength=3.0)
for name, build in [("plain LR", sa.plain_lr), ("interaction LR", sa.interaction_lr),
                    ("random forest", sa.rf), ("boosted trees", sa.gbt)]:
    o = sa.optimism(build, mk, XF, XT, n_sims=10)
    print(f"{name:14s}: train_auc={o['resub_auc']:.3f}  "
          f"cv_auc={o['cv_auc']:.3f}  optimism_delta_auc={o['optimism']:+.3f}")

**Reading it:** linear models barely overfit; both tree ensembles overfit
heavily (optimism ~0.16-0.17) and by similar amounts, so on this axis the random
forest is not clearly safer than boosting. How much the trees overfit depends on
the feature cloud, so check it under `feature_model="gmm"` and `"gaussian"` too.
The robust ordering is linear << trees; the RF-vs-boosting gap is small enough
that it should not be leaned on.

### Top-k noise: is "top-50" a stable operating point?

Across-fold spread of precision@k. Smaller k is noisier, so a too-small candidate
budget is dominated by which galaxies happened to land in which fold.

In [ ]:
for k, v in sa.k_sensitivity(sa.plain_lr, mk, XF, XT).items():
    print(f"per_fold_precision_at_{k:<2d}: mean={v['mean']:.3f}  std_across_folds={v['std']:.3f}")

## 3. Label noise: when `maser = 0` means "not detected"

A non-detection is not always a true negative: a galaxy can host a maser that was
simply too faint to see at its distance. `distance_noise=True` reproduces this, a
true maser (`z_true=1`) beyond the sensitivity limit is recorded as `target=0`.
The black x's below are those false negatives, and they are the distant ones.

In [ ]:
xcol, ycol = FEATURES["xray"]
d = sd.make_dataset("xray", scenario="wedge", strength=3.0,
                    distance_noise=True, seed=0)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

nm = d[d.z_true == 0]
seen = d[(d.z_true == 1) & (d.detected == 1)]
missed = d[(d.z_true == 1) & (d.detected == 0)]
ax[0].scatter(nm[xcol], nm[ycol], s=6, c="lightgray")
ax[0].scatter(seen[xcol], seen[ycol], s=18, c="firebrick", label="detected maser")
ax[0].scatter(missed[xcol], missed[ycol], s=40, c="black", marker="x",
              label="missed (false neg)")
ax[0].set_xlabel(xcol); ax[0].set_ylabel(ycol)
ax[0].set_title("true masers: detected vs missed")
ax[0].legend(fontsize=8)
ax[1].hist([seen["Lum_dis"], missed["Lum_dis"]], bins=18, stacked=True,
           color=["firebrick", "black"], label=["detected", "missed"])
ax[1].set_xlabel("Lum_dis (Mpc)"); ax[1].set_ylabel("count")
ax[1].set_title("missed masers are the distant ones")
ax[1].legend()
plt.tight_layout()
plt.show()

### Why it matters, and the fix

Rank the galaxies by the model and take the top 50. We can score that list three
ways: against the survey's observed labels (what you could actually compute on
real data), against the latent truth `z_true` (only possible in simulation), and
against the observed labels after a `Lum_dis < 170 Mpc` cut (the pre-registered
remedy).

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold

def oof_topk(frame, eval_col, k=50):
    X = frame[FEATURES["xray"]].to_numpy()
    y = frame[TARGET["xray"]].to_numpy()
    p = cross_val_predict(sa.plain_lr(), X, y, method="predict_proba",
                          cv=StratifiedKFold(5, shuffle=True, random_state=0))[:, 1]
    top = np.argsort(p)[::-1][:k]
    return frame[eval_col].to_numpy()[top].mean()

clean, observed, truth, cut = [], [], [], []
for s in range(20):
    cl = sd.make_dataset("xray", "wedge", strength=3.0, seed=s)
    dn = sd.make_dataset("xray", "wedge", strength=3.0, distance_noise=True, seed=s)
    near = dn[dn["Lum_dis"] < 170].reset_index(drop=True)
    clean.append(oof_topk(cl, TARGET["xray"]))
    observed.append(oof_topk(dn, TARGET["xray"]))
    truth.append(oof_topk(dn, "z_true"))
    cut.append(oof_topk(near, TARGET["xray"]))

print("Mean precision@50 across 20 simulated datasets")
print(f"no_distance_noise_reference                 : {np.mean(clean):.3f}")
print(f"distance_noise_scored_vs_observed_labels    : {np.mean(observed):.3f}  <- what real data would show")
print(f"distance_noise_scored_vs_latent_true_labels : {np.mean(truth):.3f}  <- only knowable in simulation")
print(f"distance_noise_Dlt170_scored_vs_observed    : {np.mean(cut):.3f}  <- robustness remedy")

**Reading it:** scored against the survey's labels on the full sample,
precision@50 looks worse (~0.29) than with no distance noise (~0.33). The model
has not gotten worse: scored against the latent truth it is actually ~0.36,
because it keeps ranking distant real masers highly from their features even
though the survey recorded them as non-detections. Those are exactly the
prospective candidates worth re-observing. Restricting to the well-searched
`Lum_dis < 170` subsample, where the labels are trustworthy, recovers an honest
~0.34. That is why we report results with a distance cut; it is the concrete form
of the pre-registration's D<170 robustness check.

## Where to go next

These baseline results depend on the simulator assumptions. Before any of them
is written into the frozen pre-registration, check the **Open questions to
chase** in `docs/SYNTHETIC_DATA_GUIDE.md`: bootstrap/GMM/Gaussian feature
models, sharper or non-monotone boundaries, strength and `wise_signal` sweeps,
the WISE plane, distance noise on every comparison, and calibration. A
conclusion that holds across settings is safe to commit; one that wobbles is
itself the finding.

The real labels are used once, after the plan is frozen.